# Fleet maintenance cost: v08

**v8: v5 with risk x4**

Public leaderboard: v8 18,170.52

This notebook contains all of this version's code. Each script is written to disk with `%%writefile` (so scripts can import each other) and then run with `%run`.

**Before running:** put the competition files `train.csv`, `test.csv` and `sample_submission.csv` in a `data/` folder next to this notebook, and install `numpy pandas scipy scikit-learn lightgbm xgboost catboost`.

In [ ]:
import os
os.makedirs("oof", exist_ok=True)   # saved model predictions go here

## Shared modules

### `fleet_prep.py`
Loads the data and undoes the test-depot sensor shifts.

In [ ]:
%%writefile fleet_prep.py
"""Data loading and depot-shift corrections for the Fleet Maintenance competition.

Findings from adversarial validation (train vs test AUC was 1.000 before fixes):
  * s054..s063 look like a 10-bin histogram. In test the bins are shifted one
    column to the right (test s054 is always 0 and test s055..s063 match train
    s054..s062). We shift them back and mark test s063 as missing.
  * s120 has a constant offset of ~16,025,000 in test.
  * s122 is scaled by ~2.54 in test (looks like a unit change).
  * s047 is entirely missing in test, so it is dropped.
"""
import numpy as np
import pandas as pd

HIST = [f"s{i:03d}" for i in range(54, 64)]
S120_OFFSET = 16_025_000.0
S122_SCALE = 2.54


def fix_test(te: pd.DataFrame) -> pd.DataFrame:
    te = te.copy()
    vals = {f"s{i:03d}": te[f"s{i+1:03d}"].to_numpy() for i in range(54, 63)}
    for c, v in vals.items():
        te[c] = v
    te["s063"] = np.nan
    te["s120"] = np.maximum(te["s120"] - S120_OFFSET, 0)
    te["s122"] = te["s122"] / S122_SCALE
    return te


ROUNDING = {"s055": 10, "s062": 10, "s057": 100}  # test stores these coarser


def fix_train(tr: pd.DataFrame) -> pd.DataFrame:
    tr = tr.copy()
    for c, g in ROUNDING.items():
        tr[c] = np.round(tr[c] / g) * g
    return tr


def add_features(df: pd.DataFrame, sensor_cols) -> pd.DataFrame:
    out = df[sensor_cols].copy()
    out["n_missing"] = df[sensor_cols].isna().sum(1)
    out["n_zero"] = (df[sensor_cols] == 0).sum(1)
    h = df[HIST[:-1]]  # s063 is unavailable in test
    tot = h.sum(1, min_count=1)
    out["hist_total"] = tot
    for c in HIST[:-1]:
        out[f"{c}_frac"] = df[c] / tot.replace(0, np.nan)
    w = np.arange(len(HIST) - 1)
    out["hist_mean_bin"] = (h.fillna(0).to_numpy() * w).sum(1) / tot.replace(0, np.nan)
    return out


def load(data_dir="data"):
    tr = fix_train(pd.read_csv(f"{data_dir}/train.csv"))
    te = fix_test(pd.read_csv(f"{data_dir}/test.csv"))
    sensors = [c for c in te.columns if c.startswith("s") and c not in ("s047", "s063")]
    X = add_features(tr, sensors)
    Xt = add_features(te, sensors)
    return tr, te, X, Xt

### `policy.py`
Turns failure probability + repair-cost distribution into the cost-optimal inspect/budget decision.

In [ ]:
%%writefile policy.py
"""Cost-optimal inspect/budget decisions from probabilities and cost predictions.

Cost per vehicle (competition statement):
  not inspected: C (actual repair cost, 0 if healthy)
  inspected:     10 + 0.20 * max(C - b, 0) + 0.02 * max(b - C, 0)

For a vehicle with failure probability p and conditional repair-cost distribution
C | fail ~ exp(mu + eps) (eps = empirical out-of-fold residuals), the expected
inspection cost is minimised by the 0.20/0.22 = 0.909 quantile of the mixture
(0 with prob 1-p, C|fail with prob p). We inspect only when that expected cost is
below the expected cost of not inspecting, p * E[C | fail].
"""
import numpy as np

UNDER, OVER, FIXED = 0.20, 0.02, 10.0
TAU = UNDER / (UNDER + OVER)


def competition_cost(c, flag, budget):
    return np.where(flag == 1,
                    FIXED + UNDER * np.maximum(c - budget, 0) + OVER * np.maximum(budget - c, 0),
                    c)


def decide(p, mu, eps, alpha=1.0, eps_scale=1.0, margin=0.0, budget_mult=1.0):
    """Return flag, budget arrays.

    alpha       multiplies probabilities (calibration / prior adjustment)
    eps_scale   widens or narrows the residual distribution
    margin      extra cost an inspection must save before we flag
    budget_mult scales the optimal budget (kept at 1 unless CV says otherwise)
    """
    p = np.clip(alpha * p, 1e-9, 1 - 1e-9)
    e = np.sort(eps * eps_scale)
    # samples of C|fail per vehicle: shape (n, k)
    out_flag = np.zeros(len(p), dtype=int)
    out_b = np.zeros(len(p))
    for s in range(0, len(p), 4000):
        sl = slice(s, s + 4000)
        C = np.exp(mu[sl, None] + e[None, :])
        k = C.shape[1]
        ps = p[sl]
        u = 1 - (1 - TAU) / ps  # required quantile of C|fail
        idx = np.clip(np.ceil(u * k).astype(int) - 1, 0, k - 1)
        b = np.where(u > 0, C[np.arange(C.shape[0]), idx], 0.0) * budget_mult
        loss_fail = (UNDER * np.maximum(C - b[:, None], 0) + OVER * np.maximum(b[:, None] - C, 0)).mean(1)
        insp = FIXED + ps * loss_fail + (1 - ps) * OVER * b
        no_insp = ps * C.mean(1)
        f = (no_insp - insp > margin).astype(int)
        out_flag[sl] = f
        out_b[sl] = np.where(f == 1, b, 0.0)
    return out_flag, out_b

## Pipeline (run in order)

### Step 1: `oof_models.py`
Out-of-fold failure probabilities and conditional repair-cost predictions.

In [ ]:
%%writefile oof_models.py
"""Out-of-fold failure probabilities and conditional repair-cost predictions.

Writes oof/*.npy so the decision policy can be tuned without retraining.
"""
import os, sys, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from fleet_prep import load

SEEDS = [0, 1, 2]
NF = 5
os.makedirs("oof", exist_ok=True)
tr, te, X, Xt = load()
y = tr.failure.to_numpy()
cost = tr.repair_cost.fillna(0).to_numpy()
# Censoring: healthy vehicles observed < 365 days are only partly informative.
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)


def clf_models(seed):
    return {
        "lgb": lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15,
                                  min_child_samples=40, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.5, reg_lambda=5, random_state=seed,
                                  verbosity=-1),
        "xgb": xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4,
                                 min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
                                 reg_lambda=5, tree_method="hist", random_state=seed,
                                 eval_metric="logloss", n_jobs=-1),
        "cat": CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5,
                                  l2_leaf_reg=8, random_seed=seed, verbose=False,
                                  allow_writing_files=False, thread_count=-1),
    }


def run_clf(use_w):
    tag = "w" if use_w else "nw"
    oof = {k: np.zeros(len(X)) for k in ["lgb", "xgb", "cat"]}
    tst = {k: np.zeros(len(Xt)) for k in oof}
    for seed in SEEDS:
        skf = StratifiedKFold(NF, shuffle=True, random_state=seed)
        for f, (a, b) in enumerate(skf.split(X, y)):
            for name, m in clf_models(seed).items():
                t = time.time()
                m.fit(X.iloc[a], y[a], sample_weight=w[a] if use_w else None)
                oof[name][b] += m.predict_proba(X.iloc[b])[:, 1] / len(SEEDS)
                tst[name] += m.predict_proba(Xt)[:, 1] / (len(SEEDS) * NF)
            print(f"[{tag}] seed {seed} fold {f} done", flush=True)
    for k in oof:
        np.save(f"oof/p_{k}_{tag}.npy", oof[k]); np.save(f"oof/pt_{k}_{tag}.npy", tst[k])
        print(tag, k, "AUC %.4f AP %.4f LL %.5f" % (roc_auc_score(y, oof[k]),
              average_precision_score(y, oof[k]), log_loss(y, oof[k])))
    avg = np.mean([oof[k] for k in oof], 0)
    print(tag, "avg AUC %.4f AP %.4f" % (roc_auc_score(y, avg), average_precision_score(y, avg)))


def run_cost():
    pos = np.where(y == 1)[0]
    Xp, lc = X.iloc[pos], np.log(cost[pos])
    oof = np.zeros(len(pos)); tst_all = np.zeros(len(Xt)); trn_all = np.zeros(len(X))
    for seed in SEEDS:
        for a, b in KFold(NF, shuffle=True, random_state=seed).split(Xp):
            ms = [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7,
                                    min_child_samples=10, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.5, reg_lambda=10, random_state=seed,
                                    verbosity=-1),
                  CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                    random_seed=seed, verbose=False, allow_writing_files=False)]
            for m in ms:
                m.fit(Xp.iloc[a], lc[a])
                oof[b] += m.predict(Xp.iloc[b]) / (2 * len(SEEDS))
                tst_all += m.predict(Xt) / (2 * len(SEEDS) * NF)
                trn_all += m.predict(X) / (2 * len(SEEDS) * NF)
    # For non-failed train rows use the average of fold models (they never saw them).
    trn_all[pos] = oof
    np.save("oof/mu_train.npy", trn_all); np.save("oof/mu_test.npy", tst_all)
    r = np.corrcoef(oof, lc)[0, 1]
    print("cost model: corr(oof, log cost)=%.3f, RMSE %.3f vs const %.3f" % (
        r, np.sqrt(np.mean((oof - lc) ** 2)), lc.std()))


if __name__ == "__main__":
    what = sys.argv[1:] or ["cost", "w", "nw"]
    if "cost" in what: run_cost()
    if "w" in what: run_clf(True)
    if "nw" in what: run_clf(False)

In [ ]:
%run oof_models.py cost w

### Step 2: `full_fit.py`
Fit the ensemble on all training data; predict test both as-is and after quantile-matching each test sensor to the training distribution (removes the remaining depot-level scale shift in the heavy-usage tail).

In [ ]:
%%writefile full_fit.py
"""Fit the ensemble on all training data; predict test both as-is and after
quantile-matching each test sensor to the training distribution (removes the
remaining depot-level scale shift in the heavy-usage tail)."""
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from fleet_prep import load, add_features

def qmatch(train_col, test_col):
    a = np.sort(train_col.dropna().to_numpy()); t = test_col.copy()
    m = t.notna()
    r = t[m].rank(pct=True, method="average").to_numpy()
    t[m] = np.quantile(a, np.clip(r - 0.5 / m.sum(), 0, 1))
    return t

tr, te, X, Xt = load()
sensors = [c for c in Xt.columns if c.startswith("s") and "_" not in c]
te_q = te.copy()
for c in sensors:
    te_q[c] = qmatch(tr[c], te[c])
Xq = add_features(te_q, sensors)
y = tr.failure.to_numpy(); cost = tr.repair_cost.fillna(0).to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
P, PQ = [], []
for seed in [0, 1]:
    ms = [lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                             random_state=seed, verbosity=-1),
          xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4, min_child_weight=3,
                            subsample=0.8, colsample_bytree=0.5, reg_lambda=5, tree_method="hist",
                            random_state=seed, n_jobs=-1),
          CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8,
                             random_seed=seed, verbose=False, allow_writing_files=False)]
    for m in ms:
        m.fit(X, y, sample_weight=w)
        P.append(m.predict_proba(Xt)[:, 1]); PQ.append(m.predict_proba(Xq)[:, 1])
pos = y == 1
MU, MUQ = [], []
for seed in [0, 1]:
    for m in [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=10,
                                random_state=seed, verbosity=-1),
              CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X[pos], np.log(cost[pos])); MU.append(m.predict(Xt)); MUQ.append(m.predict(Xq))
np.save("oof/full_pt.npy", np.mean(P, 0)); np.save("oof/full_ptq.npy", np.mean(PQ, 0))
np.save("oof/full_mut.npy", np.mean(MU, 0)); np.save("oof/full_mutq.npy", np.mean(MUQ, 0))
print("mean test p: as-is %.4f  quantile-matched %.4f  (train rate %.4f)" % (np.mean(P), np.mean(PQ), y.mean()))

In [ ]:
%run full_fit.py

### Step 3

In [ ]:
# v8_more_aggressive: full-data ensemble with test-depot risk multiplier x4.0, budgets x1.0
import numpy as np, pandas as pd
from policy import decide
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
f, b = decide(np.load("oof/full_pt.npy"), np.load("oof/full_mut.npy"), eps, 4.0, 1.0, 0.0, 1.0)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
sub.to_csv("submission_v8_more_aggressive.csv", index=False)
print("v8_more_aggressive: inspections", int(f.sum()))

## Output files

In [ ]:
import glob, pandas as pd
for p in sorted(glob.glob("submission_*.csv")):
    s = pd.read_csv(p)
    print(f"{p}: {int(s.flag.sum())} inspections, mean budget {s.budget[s.flag == 1].mean():.0f}")

## Other scripts in this version
Experiments and tuning scripts that are not needed for this version's submission. Shown for completeness; they are written to disk but not run.

### `make_submission.py`
Build the final submission from saved ensemble predictions (run oof_models.py first).

In [ ]:
%%writefile make_submission.py
"""Build the final submission from saved ensemble predictions (run oof_models.py first)."""
import numpy as np, pandas as pd
from policy import decide

ALPHA, EPS_SCALE, MARGIN = 1.5, 1.0, 2.0   # chosen on out-of-fold cost (0.838 vs 3.49 do-nothing)
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); mut = np.load("oof/mu_test.npy")
eps = np.log(c[y == 1]) - mu[y == 1]
pt = np.mean([np.load(f"oof/pt_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
f, b = decide(pt, mut, eps, ALPHA, EPS_SCALE, MARGIN)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
samp = pd.read_csv("data/sample_submission.csv")
assert list(sub.columns) == list(samp.columns) and (sub.id.values == samp.id.values).all()
assert sub.flag.isin([0, 1]).all() and np.isfinite(sub.budget).all() and (sub.budget >= 0).all()
assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
sub.to_csv("submission_v5_ensemble.csv", index=False)
print("flagged", sub.flag.sum(), "of", len(sub), "| median budget", sub.loc[sub.flag == 1, "budget"].median())

### `make_variants.py`
Write submission variants from the full-data fit (run full_fit.py first).

In [ ]:
%%writefile make_variants.py
"""Write submission variants from the full-data fit (run full_fit.py first)."""
import numpy as np, pandas as pd
from policy import decide

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
V = {
    # quantile-matched test sensors, same policy settings as v5
    "v6_qmatch": (np.load("oof/full_ptq.npy"), np.load("oof/full_mutq.npy"), 1.5, 1.0, 2.0),
    # v5 inputs but more inspections (use if v6 scores worse than v5)
    "v7_aggressive": (np.load("oof/full_pt.npy"), np.load("oof/full_mut.npy"), 2.5, 1.0, 0.0),
}
for name, (p, m, a, s, mg) in V.items():
    f, b = decide(p, m, eps, a, s, mg)
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all() and len(sub) == 20000
    sub.to_csv(f"submission_{name}.csv", index=False)
    fl = sub.flag == 1
    print(f"{name}: flags={fl.sum()}  sum(10+0.02*budget)={10*fl.sum()+0.02*sub.budget[fl].sum():.0f}")

### `tune.py`
Tune the decision policy on out-of-fold predictions and compare to baselines.

In [ ]:
%%writefile tune.py
"""Tune the decision policy on out-of-fold predictions and compare to baselines."""
import itertools, sys
import numpy as np
import pandas as pd
from policy import decide, competition_cost

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost", "followup_days"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
pos = y == 1
mu = np.load("oof/mu_train.npy")
eps = np.log(c[pos]) - mu[pos]


def P(tag, names=("lgb", "xgb", "cat")):
    return np.mean([np.load(f"oof/p_{n}_{tag}.npy") for n in names], 0)


def score(flag, b):
    return competition_cost(c, flag, b).mean()


print("Do nothing (never inspect):          %.4f" % c.mean())
q90 = np.quantile(c[pos], 0.9)
for tag in ["w", "nw"]:
    for names in [("lgb",), ("xgb",), ("cat",), ("lgb", "xgb", "cat")]:
        p = P(tag, names)
        # notebook-style baseline: threshold on p*exp(mu), fixed q90 budget
        e = p * np.exp(mu)
        best = min(score((e > t).astype(int), np.where(e > t, q90, 0))
                   for t in np.quantile(e, np.linspace(0.9, 0.9995, 200)))
        f, b = decide(p, mu, eps)
        print(f"{tag:>2} {'+'.join(names):12s} threshold+q90 best={best:.4f} | "
              f"optimal policy (no tuning)={score(f, b):.4f} flags={f.sum()}")

tag = sys.argv[1] if len(sys.argv) > 1 else "w"
p = P(tag)
rows = []
for a, s, m, bm in itertools.product([0.6, 0.8, 1.0, 1.2, 1.5], [0.8, 1.0, 1.2],
                                     [-2, 0, 2, 5], [0.8, 1.0, 1.2]):
    f, b = decide(p, mu, eps, a, s, m, bm)
    rows.append((a, s, m, bm, score(f, b), f.sum()))
r = pd.DataFrame(rows, columns=["alpha", "eps_scale", "margin", "budget_mult", "cost", "flags"])
print(r.sort_values("cost").head(10).to_string(index=False))